# Chapter 3: Dimensionality Reduction Techniques

## Ringkasan Bab & Penjelasan Teoretis
Reduksi dimensi bertujuan mengurangi jumlah fitur pada dataset berdimensi tinggi sambil mempertahankan informasi penting sebanyak mungkin. Manfaat utamanya meliputi penyederhanaan data, pengurangan biaya komputasi, pencegahan *overfitting* (*curse of dimensionality*), dan visualisasi ruang berdimensi tinggi ke dalam 2D atau 3D.

### 1. Principal Component Analysis (PCA)
**PCA** adalah metode ekstraksi fitur **unsupervised** yang mentransformasikan data ke sistem koordinat baru (*principal components*) yang saling ortogonal dan mengarah pada varians terbesar di dalam data.
* **Tahapan Matematis**: Standardisasi data ($\mu=0, \sigma=1$), perhitungan matriks kovarians, perhitungan *eigenvalues* (besar varians) dan *eigenvectors* (arah komponen utama), lalu memproyeksikan data ke komponen utama teratas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_digits, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
import warnings

np.random.seed(2024)
warnings.simplefilter(action='ignore', category=FutureWarning)

wine = load_wine()
df_wine = pd.DataFrame(data=wine.data, columns=wine.feature_names)
target_wine = wine.target

pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2))
])

X_pca = pca_pipeline.fit_transform(df_wine)

plt.figure(figsize=(10, 8))
shapes = ['o', '^', 'D']
colors = ['r', 'g', 'b']

for i, (shape, color) in enumerate(zip(shapes, colors)):
    plt.scatter(
        X_pca[target_wine == i, 0], X_pca[target_wine == i, 1],
        c=color, marker=shape, label=wine.target_names[i]
    )

pca = pca_pipeline.named_steps['pca']
origin = np.zeros(2)
arrow_colors = ['black', 'orange']
scaling = 3

for i, (component, ratio) in enumerate(zip(pca.components_, pca.explained_variance_ratio_)):
    plt.arrow(
        origin[0], origin[1],
        component[0] * scaling, component[1] * scaling,
        color=arrow_colors[i], width=0.02, head_width=0.2, head_length=0.2,
        label=f'PC{i+1} ({ratio:.1%} variance)'
    )

plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Wine Dataset - First Two Principal Components')
plt.legend(title="Classes")
plt.grid(True, alpha=0.3)
plt.show()

# Plot Explained Variance Ratio
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.sum(explained_variance_ratio)

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(1, len(explained_variance_ratio) + 1)
bars = ax.bar(x, explained_variance_ratio)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2%}', ha='center', va='bottom')

ax.text(0.95, 0.95, f'Total Variance\nExplained: {cumulative_variance:.2%}',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
ax.set_xticks(x)
ax.set_xticklabels(['PC1', 'PC2'])
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('Explained Variance Ratio by Principal Component')
plt.show()

### 2. Linear Discriminant Analysis (LDA) & Perbandingannya dengan PCA
**LDA** adalah teknik reduksi dimensi **supervised** yang memanfaatkan label kelas untuk mencari kombinasi linear fitur (*linear discriminants*) yang **memaksimalkan pemisahan antar-kelas** sekaligus **meminimalkan varians di dalam masing-masing kelas**. Jumlah komponen LDA dibatasi hingga $\min(\text{n\_features}, \text{n\_classes} - 1)$.

In [ ]:
X_wine, y_wine = wine.data, wine.target

lda_pipeline_wine = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis(n_components=2))
])

X_lda_wine = lda_pipeline_wine.fit_transform(X_wine, y_wine)

plt.figure(figsize=(14, 6))
plt.subplot(121)
for i, (shape, color) in enumerate(zip(shapes, colors)):
    mask = y_wine == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, marker=shape, edgecolor='black', label=wine.target_names[i])
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Wine Dataset - PCA Components')
plt.legend()

plt.subplot(122)
for i, (shape, color) in enumerate(zip(shapes, colors)):
    mask = y_wine == i
    plt.scatter(X_lda_wine[mask, 0], X_lda_wine[mask, 1], c=color, marker=shape, edgecolor='black', label=wine.target_names[i])
plt.xlabel('First LDA Component')
plt.ylabel('Second LDA Component')
plt.title('Wine Dataset - LDA Components')
plt.legend()

plt.tight_layout()
plt.show()

### 3. t-Distributed Stochastic Neighbor Embedding (t-SNE) & Latihan Praktis
**t-SNE** adalah teknik reduksi dimensi **non-linear** yang mengubah jarak Euclidean antar-titik menjadi probabilitas kondisional untuk mempertahankan struktur klaster lokal saat memetakan data berdimensi tinggi ke ruang visual 2D atau 3D.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

digits = load_digits()
tsne_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('tsne', TSNE(n_components=2, random_state=2024))
])

X_tsne = tsne_pipeline.fit_transform(digits.data)

plt.figure(figsize=(10, 8))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=digits.target, cmap='Paired')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('Digits Dataset - t-SNE Visualization')
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.Paired(i/9), label=str(i), markersize=10)
    for i in range(10)
]
plt.legend(handles=legend_elements, title='Digit', loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

# Practical Exercise: Perbandingan Logistic Regression Tanpa PCA vs Dengan PCA pada Iris Dataset
iris = load_iris()
X_ir_train, X_ir_test, y_ir_train, y_ir_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=2024)

pipe_no_pca = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]).fit(X_ir_train, y_ir_train)
pipe_with_pca = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=2)), ('clf', LogisticRegression())]).fit(X_ir_train, y_ir_train)

print(f"Akurasi Tanpa PCA (4 Fitur): {accuracy_score(y_ir_test, pipe_no_pca.predict(X_ir_test)):.4f}")
print(f"Akurasi Dengan PCA (2 Komponen): {accuracy_score(y_ir_test, pipe_with_pca.predict(X_ir_test)):.4f}")